In [41]:
from datetime import datetime, timezone
from garminconnect import Garmin
from notion_client import Client
from dotenv import load_dotenv
import pytz
import os

# Your local time zone, replace with the appropriate one if needed
local_tz = pytz.timezone('America/Toronto')

In [42]:
ACTIVITY_ICONS = {
    "Barre": "https://img.icons8.com/?size=100&id=66924&format=png&color=000000",
    "Breathwork": "https://img.icons8.com/?size=100&id=9798&format=png&color=000000",
    "Cardio": "https://img.icons8.com/?size=100&id=71221&format=png&color=000000",
    "Cycling": "https://img.icons8.com/?size=100&id=47443&format=png&color=000000",
    "Hiking": "https://img.icons8.com/?size=100&id=9844&format=png&color=000000",
    "Indoor Cardio": "https://img.icons8.com/?size=100&id=62779&format=png&color=000000",
    "Indoor Cycling": "https://img.icons8.com/?size=100&id=47443&format=png&color=000000",
    "Indoor Rowing": "https://img.icons8.com/?size=100&id=71098&format=png&color=000000",
    "Pilates": "https://img.icons8.com/?size=100&id=9774&format=png&color=000000",
    "Meditation": "https://img.icons8.com/?size=100&id=9798&format=png&color=000000",
    "Rowing": "https://img.icons8.com/?size=100&id=71491&format=png&color=000000",
    "Running": "https://img.icons8.com/?size=100&id=k1l1XFkME39t&format=png&color=000000",
    "Strength Training": "https://img.icons8.com/?size=100&id=107640&format=png&color=000000",
    "Stretching": "https://img.icons8.com/?size=100&id=djfOcRn1m_kh&format=png&color=000000",
    "Swimming": "https://img.icons8.com/?size=100&id=9777&format=png&color=000000",
    "Treadmill Running": "https://img.icons8.com/?size=100&id=9794&format=png&color=000000",
    "Walking": "https://img.icons8.com/?size=100&id=9807&format=png&color=000000",
    "Yoga": "https://img.icons8.com/?size=100&id=9783&format=png&color=000000",
    # Add more mappings as needed
}

In [43]:
def get_all_activities(garmin, limit=1000):
    return garmin.get_activities(0, limit)

In [44]:
def format_activity_type(activity_type, activity_name=""):
    # First format the activity type as before
    formatted_type = activity_type.replace('_', ' ').title() if activity_type else "Unknown"

    # Initialize subtype as the same as the main type
    activity_subtype = formatted_type
    activity_type = formatted_type

    # Map of specific subtypes to their main types
    activity_mapping = {
        "Barre": "Strength",
        "Indoor Cardio": "Cardio",
        "Indoor Cycling": "Cycling",
        "Indoor Rowing": "Rowing",
        "Speed Walking": "Walking",
        "Strength Training": "Strength",
        "Treadmill Running": "Running"
    }

    # Special replacement for Rowing V2
    if formatted_type == "Rowing V2":
        activity_type = "Rowing"

    # Special case for Yoga and Pilates
    elif formatted_type in ["Yoga", "Pilates"]:
        activity_type = "Yoga/Pilates"
        activity_subtype = formatted_type

    # If the formatted type is in our mapping, update both main type and subtype
    if formatted_type in activity_mapping:
        activity_type = activity_mapping[formatted_type]
        activity_subtype = formatted_type

    # Special cases for activity names
    if activity_name and "meditation" in activity_name.lower():
        return "Meditation", "Meditation"
    if activity_name and "barre" in activity_name.lower():
        return "Strength", "Barre"
    if activity_name and "stretch" in activity_name.lower():
        return "Stretching", "Stretching"
    
    return activity_type, activity_subtype

In [45]:
def format_entertainment(activity_name):
    return activity_name.replace('ENTERTAINMENT', 'Netflix')

In [46]:
def format_training_message(message):
    messages = {
        'NO_': 'No Benefit',
        'MINOR_': 'Some Benefit',
        'RECOVERY_': 'Recovery',
        'MAINTAINING_': 'Maintaining',
        'IMPROVING_': 'Impacting',
        'IMPACTING_': 'Impacting',
        'HIGHLY_': 'Highly Impacting',
        'OVERREACHING_': 'Overreaching'
    }
    for key, value in messages.items():
        if message.startswith(key):
            return value
    return message

In [48]:
def format_training_effect(trainingEffect_label):
    return trainingEffect_label.replace('_', ' ').title()

In [49]:
def format_pace(average_speed):
    if average_speed > 0:
        pace_min_km = 1000 / (average_speed * 60)  # Convert to min/km
        minutes = int(pace_min_km)
        seconds = int((pace_min_km - minutes) * 60)
        total_seconds = minutes * 60 + seconds
        conversion_seconds = total_seconds * 1.61
        converted_minutes = int(conversion_seconds / 60)
        converted_seconds = int(conversion_seconds%60)
        return f"{converted_minutes}:{converted_seconds:02d} min/mi"
    else:
        return ""

In [50]:
def activity_exists(client, database_id, activity_date, activity_type, activity_name):

    # Check if an activity already exists in the Notion database and return it if found.

    # Handle the activity_type which is now a tuple
    if isinstance(activity_type, tuple):
        main_type, _ = activity_type
    else:
        main_type = activity_type[0] if isinstance(activity_type, (list, tuple)) else activity_type
    
    # Determine the correct activity type for the lookup
    lookup_type = "Stretching" if "stretch" in activity_name.lower() else main_type
    
    query = client.databases.query(
        database_id=database_id,
        filter={
            "and": [
                {"property": "Date", "date": {"equals": activity_date.split('T')[0]}},
                {"property": "Activity Type", "select": {"equals": lookup_type}},
                {"property": "Activity Name", "title": {"equals": activity_name}}
            ]
        }
    )
    results = query['results']
    return results[0] if results else None

In [51]:
def activity_needs_update(existing_activity, new_activity):
    existing_props = existing_activity['properties']
    
    activity_name = new_activity.get('activityName', '').lower()
    activity_type, activity_subtype = format_activity_type(
        new_activity.get('activityType', {}).get('typeKey', 'Unknown'),
        activity_name
    )
    
    # Check if 'Subactivity Type' property exists
    has_subactivity = (
        'Subactivity Type' in existing_props and 
        existing_props['Subactivity Type'] is not None and
        existing_props['Subactivity Type'].get('select') is not None
    )
    
    return (
        existing_props['Distance (mi)']['number'] != round(new_activity.get('distance', 0) / 1609.344, 2) or
        existing_props['Duration (min)']['number'] != round(new_activity.get('duration', 0) / 60, 2) or
        existing_props['Calories']['number'] != round(new_activity.get('calories', 0)) or
        existing_props['Avg Pace']['rich_text'][0]['text']['content'] != format_pace(new_activity.get('averageSpeed', 0)) or
        existing_props['Avg Power']['number'] != round(new_activity.get('avgPower', 0), 1) or
        existing_props['Max Power']['number'] != round(new_activity.get('maxPower', 0), 1) or
        existing_props['Training Effect']['select']['name'] != format_training_effect(new_activity.get('trainingEffectLabel', 'Unknown')) or
        existing_props['Aerobic']['number'] != round(new_activity.get('aerobicTrainingEffect', 0), 1) or
        existing_props['Aerobic Effect']['select']['name'] != format_training_message(new_activity.get('aerobicTrainingEffectMessage', 'Unknown')) or
        existing_props['Anaerobic']['number'] != round(new_activity.get('anaerobicTrainingEffect', 0), 1) or
        existing_props['Anaerobic Effect']['select']['name'] != format_training_message(new_activity.get('anaerobicTrainingEffectMessage', 'Unknown')) or
        existing_props['PR']['checkbox'] != new_activity.get('pr', False) or
        existing_props['Fav']['checkbox'] != new_activity.get('favorite', False) or
        existing_props['Activity Type']['select']['name'] != activity_type or
        (has_subactivity and existing_props['Subactivity Type']['select']['name'] != activity_subtype) or
        (not has_subactivity)  # If the property doesn't exist, we need an update
    )

In [85]:
def format_activity_date(activity_date):
    # Ensure the date format is correct
    # Create GMT timezone object
    gmt_tz = pytz.timezone('GMT')
    
    # Parse the input GMT time string (example format: '2023-03-07T14:30:00Z')
    gmt_time = datetime.strptime(activity_date, '%Y-%m-%d %H:%M:%S')

    # Localize the GMT time to make it timezone-aware
    gmt_time = gmt_tz.localize(gmt_time)

    # Create EST timezone object (Eastern Time which handles both EST and EDT)
    est_tz = pytz.timezone('US/Eastern')

    # Convert the time to EST (this will automatically adjust for DST if needed)
    est_time = gmt_time.astimezone(est_tz)

    est_time = est_time.replace(second=0)

    return est_time.strftime('%Y-%m-%d %H:%M:%S')


In [53]:
def create_activity(client, database_id, activity):

    # Create a new activity in the Notion database
    activity_date = format_activity_date(activity.get('startTimeGMT'))
    activity_name = format_entertainment(activity.get('activityName', 'Unnamed Activity'))
    activity_type, activity_subtype = format_activity_type(
        activity.get('activityType', {}).get('typeKey', 'Unknown'),
        activity_name
    )
    
    # Get icon for the activity type
    icon_url = ACTIVITY_ICONS.get(activity_subtype if activity_subtype != activity_type else activity_type)
    
    properties = {
        "Date": {"date": {"start": activity_date}},
        "Activity Type": {"select": {"name": activity_type}},
        "Subactivity Type": {"select": {"name": activity_subtype}},
        "Activity Name": {"title": [{"text": {"content": activity_name}}]},
        "Distance (mi)": {"number": round(activity.get('distance', 0) / 1609.344, 2)},  # the default is in meters, divide to get to miles
        "Duration (min)": {"number": round(activity.get('duration', 0) / 60, 2)},
        "Calories": {"number": round(activity.get('calories', 0))},
        "Avg Pace": {"rich_text": [{"text": {"content": format_pace(activity.get('averageSpeed', 0))}}]},
        "Avg Power": {"number": round(activity.get('avgPower', 0), 1)},
        "Max Power": {"number": round(activity.get('maxPower', 0), 1)},
        "Training Effect": {"select": {"name": format_training_effect(activity.get('trainingEffectLabel', 'Unknown'))}},
        "Aerobic": {"number": round(activity.get('aerobicTrainingEffect', 0), 1)},
        "Aerobic Effect": {"select": {"name": format_training_message(activity.get('aerobicTrainingEffectMessage', 'Unknown'))}},
        "Anaerobic": {"number": round(activity.get('anaerobicTrainingEffect', 0), 1)},
        "Anaerobic Effect": {"select": {"name": format_training_message(activity.get('anaerobicTrainingEffectMessage', 'Unknown'))}},
        "PR": {"checkbox": activity.get('pr', False)},
        "Fav": {"checkbox": activity.get('favorite', False)}
    }
    
    page = {
        "parent": {"database_id": database_id},
        "properties": properties,
    }
    
    if icon_url:
        page["icon"] = {"type": "external", "external": {"url": icon_url}}
    
    client.pages.create(**page)

In [74]:
def format_date_from_notion(date_to_format):
    # Ensure the date format is correct
    # Create GMT timezone object
    gmt_tz = pytz.timezone('GMT')
    
    # Parse the input GMT time string (example format: '2025-06-03T22:38:00.000+00:00'')
    gmt_time = datetime.strptime(date_to_format, '%Y-%m-%dT%H:%M:%S.%f%z')

    # Create EST timezone object (Eastern Time which handles both EST and EDT)
    est_tz = pytz.timezone('US/Eastern')

    # Convert the time to EST (this will automatically adjust for DST if needed)
    est_time = gmt_time.astimezone(est_tz)
    
    return est_time.strftime('%Y-%m-%d %H:%M:%S')

In [70]:
def get_existing_activities(client, database_id):
    query = client.databases.query(database_id=database_id)
    existing_activities = {}

    # Store activities in a dictionary for easy lookup
    for result in query['results']:
        activity_name = result['properties']['Activity Name']['title'][0]['text']['content']
        activity_date = result['properties']['Date']['date']['start']
        activity_date = format_date_from_notion(activity_date)
        activity_type = result['properties']['Activity Type']['select']['name']

        existing_activities[(activity_date, activity_name, activity_type)] = result

    return existing_activities

In [55]:
def update_activity(client, existing_activity, new_activity):

    # Update an existing activity in the Notion database with new data
    activity_name = new_activity.get('activityName', 'Unnamed Activity')
    activity_type, activity_subtype = format_activity_type(
        new_activity.get('activityType', {}).get('typeKey', 'Unknown'),
        activity_name
    )
    
    # Get icon for the activity type
    icon_url = ACTIVITY_ICONS.get(activity_subtype if activity_subtype != activity_type else activity_type)
    
    properties = {
        "Activity Type": {"select": {"name": activity_type}},
        "Subactivity Type": {"select": {"name": activity_subtype}},
        "Distance (mi)": {"number": round(new_activity.get('distance', 0) / 1609.344, 2)},
        "Duration (min)": {"number": round(new_activity.get('duration', 0) / 60, 2)},
        "Calories": {"number": round(new_activity.get('calories', 0))},
        "Avg Pace": {"rich_text": [{"text": {"content": format_pace(new_activity.get('averageSpeed', 0))}}]},
        "Avg Power": {"number": round(new_activity.get('avgPower', 0), 1)},
        "Max Power": {"number": round(new_activity.get('maxPower', 0), 1)},
        "Training Effect": {"select": {"name": format_training_effect(new_activity.get('trainingEffectLabel', 'Unknown'))}},
        "Aerobic": {"number": round(new_activity.get('aerobicTrainingEffect', 0), 1)},
        "Aerobic Effect": {"select": {"name": format_training_message(new_activity.get('aerobicTrainingEffectMessage', 'Unknown'))}},
        "Anaerobic": {"number": round(new_activity.get('anaerobicTrainingEffect', 0), 1)},
        "Anaerobic Effect": {"select": {"name": format_training_message(new_activity.get('anaerobicTrainingEffectMessage', 'Unknown'))}},
        "PR": {"checkbox": new_activity.get('pr', False)},
        "Fav": {"checkbox": new_activity.get('favorite', False)}
    }
    
    update = {
        "page_id": existing_activity['id'],
        "properties": properties,
    }
    
    if icon_url:
        update["icon"] = {"type": "external", "external": {"url": icon_url}}
        
    client.pages.update(**update)

In [56]:
load_dotenv()

True

In [57]:
# Initialize Garmin and Notion clients using environment variables
garmin_email = os.getenv("GARMIN_EMAIL")
garmin_password = os.getenv("GARMIN_PASSWORD")
notion_token = os.getenv("NOTION_TOKEN")
database_id = os.getenv("NOTION_DB_ID")

In [58]:
garmin = Garmin(garmin_email, garmin_password)
garmin.login()
client = Client(auth=notion_token)

In [83]:
existing_activities = get_existing_activities(client, database_id)

In [60]:
activities = get_all_activities(garmin)

In [87]:
activity = activities[0]
activity_date = format_activity_date(activity.get('startTimeGMT'))
activity_name = format_entertainment(activity.get('activityName', 'Unnamed Activity'))
activity_type, activity_subtype = format_activity_type(
    activity.get('activityType', {}).get('typeKey', 'Unknown'),
    activity_name
)
print(activity_date)
print((activity_date, activity_name, activity_type) in existing_activities.keys())
print(existing_activities.keys())


2026-03-07 10:00:00
True
dict_keys([('2026-03-05 17:17:00', 'Base', 'Running'), ('2026-02-27 18:06:00', 'Cincinnati - Base', 'Running'), ('2026-03-07 10:00:00', 'Cincinnati - Bockfest 5K', 'Running'), ('2026-02-26 18:30:00', 'Core Stability 2', 'Strength'), ('2026-03-06 17:42:00', 'Cincinnati - Sprint', 'Running'), ('2026-03-04 18:25:00', 'Threshold', 'Running')])


In [21]:
for activity in activities:
    activity_date = format_activity_date(activity.get('startTimeGMT'))
    activity_name = format_entertainment(activity.get('activityName', 'Unnamed Activity'))
    activity_type, activity_subtype = format_activity_type(
        activity.get('activityType', {}).get('typeKey', 'Unknown'),
        activity_name
    )
    
    # Check if activity already exists in the fetched activities
    if (activity_date, activity_name, activity_type) in existing_activities:
        existing_activity = existing_activities[(activity_date, activity_name, activity_type)]
        if activity_needs_update(existing_activity, activity):
            update_activity(client, existing_activity, activity)
            print(f"Updated: {activity_type} - {activity_name}")
    else:
        create_activity(client, database_id, activity)
        print(f"Created: {activity_type} - {activity_name}")

Created: Running - Cincinnati - Bockfest 5K
Created: Running - Cincinnati - Sprint
Created: Running - Base
Created: Running - Threshold


KeyboardInterrupt: 